# Map Visualization

## Goal

Visualize resolved locations on an interactive Folium map.

## What you will do

- Load resolved outputs if available.
- Validate coordinates.
- Create an interactive map.
- Save it to `outputs/maps/sample_geoparsing_map.html`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.config import RESULTS_DIR, MAPS_DIR
from src.data_utils import load_dataframe_if_exists
from src.visualization_utils import validate_coordinates, create_location_map, save_map


def has_useful_values(series):
    cleaned = series.dropna().astype(str).str.strip()
    if cleaned.empty:
        return False
    return not cleaned[~cleaned.str.lower().isin(["", "nan", "none", "null"])].empty


def useful_optional_columns(df, columns):
    return [col for col in columns if col in df.columns and has_useful_values(df[col])]


def compact_location_table(df):
    ordered = []
    for col in ["mention", "selected_name", "name", "lat", "lon", "application_context", "method"]:
        if col in df.columns:
            ordered.append(col)
    for col in useful_optional_columns(df, ["country", "score", "confidence"]):
        if col not in ordered:
            ordered.append(col)
    return df[ordered] if ordered else df


def popup_columns_for_map(df):
    base = [col for col in ["mention", "selected_name", "name"] if col in df.columns]
    optional = useful_optional_columns(df, ["application_context", "method"])
    return base + [col for col in optional if col not in base]


## Step 1: Load available result files

In [2]:
candidate_files = [
    RESULTS_DIR / "unitoprank_results.csv",
    RESULTS_DIR / "llm_rag_results.csv",
    RESULTS_DIR / "geocoder_candidates.csv",
]
df = None
for path in candidate_files:
    df = load_dataframe_if_exists(path)
    if df is not None and not df.empty:
        print("Loaded:", path)
        break
if df is None or df.empty:
    df = pd.DataFrame([
        {"mention": "Paris", "selected_name": "Paris", "lat": 48.8566, "lon": 2.3522, "method": "built_in_example"},
        {"mention": "Berlin", "selected_name": "Berlin", "lat": 52.52, "lon": 13.405, "method": "built_in_example"},
    ])
    print("No result CSV found. Showing a tiny built-in example so the map workflow can still run.")

print("Rows loaded:", len(df))
display(compact_location_table(df).head(10))


Loaded: /home/hu_xk/Workplace/wawopensearch3_hackathon/module2_geoparsing/outputs/results/unitoprank_results.csv
Rows loaded: 3


,mention,selected_name,lat,lon,method
0,Passau,"Passau, Lower Bavaria, Bavaria, Germany, Europe",48.57318,13.45060,unitoprank_geonames_photon
1,Danube River,"Danube River, Romania, Europe",45.33333,29.66667,unitoprank_geonames_photon
2,Bavaria,"Bavaria, Germany, Europe",49.00000,11.50000,unitoprank_geonames_photon


## Step 2: Validate coordinates

In [3]:
valid = validate_coordinates(df)
print("Rows with valid coordinates:", len(valid))
display(compact_location_table(valid).head(10))


Rows with valid coordinates: 3


,mention,selected_name,lat,lon,method
0,Passau,"Passau, Lower Bavaria, Bavaria, Germany, Europe",48.57318,13.45060,unitoprank_geonames_photon
1,Danube River,"Danube River, Romania, Europe",45.33333,29.66667,unitoprank_geonames_photon
2,Bavaria,"Bavaria, Germany, Europe",49.00000,11.50000,unitoprank_geonames_photon


## Step 3: Choose popup columns

Popups should show enough context to interpret the point without making the marker hard to read.

In [4]:
popup_cols = popup_columns_for_map(valid)
print("Popup columns:", popup_cols)


Popup columns: ['mention', 'selected_name', 'method']


## Step 4: Create the map

In [5]:
fmap = create_location_map(valid, popup_cols=popup_cols)
display(fmap)


## Step 5: Save the map

In [ ]:
out = save_map(fmap, MAPS_DIR / 'sample_geoparsing_map.html')
print('Saved:', out)

## Common issues

- Rows without valid latitude and longitude are skipped.
- If there are no valid coordinates, the map opens at a world view.
- Folium maps are saved as standalone HTML files.